# Implementación de Kubernetes para ejecutar un ETL paralelo
**Proyecto:** ETL de datos de COVID-19 e influenza en México  

## Objetivo
En esta parte del proyecto se ejecutará, mediante Kubernetes, el ETL paralelo que desarrolló el Integrante 2. El objetivo es preparar un clúster local, desplegar un Job, comprobar que el proceso termina correctamente y recuperar los archivos de salida.

**Alcance:** Kubernetes administra la ejecución del contenedor. No convierte automáticamente el ETL en un procesamiento distribuido entre varios nodos; el paralelismo interno sigue siendo responsabilidad del programa Python.

## 1. Fundamento teórico
### 1.1 ¿Qué es Docker?
Docker permite empaquetar una aplicación con sus bibliotecas y configuraciones en una **imagen**. Cuando se ejecuta esa imagen, se crea un **contenedor**. En nuestro caso, la imagen contiene Python, las dependencias y el script `etl_paralelo.py`.

### 1.2 ¿Qué es Kubernetes?
Kubernetes es un sistema para administrar aplicaciones que se ejecutan en contenedores. Permite indicar qué contenedor debe iniciarse, consultar su estado, revisar registros y administrar recursos como el almacenamiento.

### 1.3 Conceptos a utilizar
- **Clúster:** conjunto de recursos administrados por Kubernetes.
- **Nodo:** máquina donde se ejecutan las cargas de trabajo.
- **Pod:** unidad mínima de ejecución; contiene uno o más contenedores.
- **Job:** recurso diseñado para ejecutar una tarea hasta que termine correctamente.
- **PVC (PersistentVolumeClaim):** solicitud de almacenamiento que puede conservar los resultados después de que termine el contenedor.
- **kubectl:** herramienta para enviar instrucciones a Kubernetes.
- **Minikube:** herramienta para crear un clúster Kubernetes local para prácticas.

### 1.4 ¿Por qué usar un Job?
Un ETL no necesita estar ejecutándose todo el tiempo: lee datos, los transforma, guarda resultados y termina. Por eso, para este proyecto usaré un **Job** en lugar de un Deployment, que está orientado a servicios de larga duración.

## 2. Herramientas y requisitos
La instalación de Kubernetes es independiente de las bibliotecas del ETL.

- Minikube: https://minikube.sigs.k8s.io/docs/start/
- kubectl: https://kubernetes.io/docs/tasks/tools/install-kubectl-linux/
- Docker: https://docs.docker.com/engine/install/ubuntu/

## 3. Iniciar el clúster local
Iniciar Kubernetes mediante Minikube utilizando Docker como controlador:
```bash
minikube start --driver=docker
kubectl get nodes
```
Debe mostrar al menos un nodo en estado `Ready`. Esto significa que Kubernetes está disponible para recibir tareas.

En la carpeta principal del proyecto ejecutar:
```bash
docker build -t etl-covid-paralelo:latest .
minikube image load etl-covid-paralelo:latest
```

**¿Qué ocurre aquí?** `docker build` crea la imagen a partir del Dockerfile. `minikube image load` hace que esa imagen esté disponible para el clúster local. El manifiesto del Job usa `imagePullPolicy: Never`, por lo que no intentará descargarla de un registro externo.

### Archivos YAML
Los manifiestos YAML describen recursos de Kubernetes:

1. `k8s/pvc-resultados.yaml`: solicita almacenamiento para guardar las salidas del ETL.
2. `k8s/job-etl.yaml`: define el contenedor que ejecutará el ETL y monta el almacenamiento.
3. `k8s/pod-lector.yaml`: permite consultar y copiar los resultados cuando el Job ya terminó.

**Diferencia importante:** el Job ejecuta el ETL; el Pod lector no vuelve a procesar los datos, solo sirve para acceder a los archivos.

### Crear el almacenamiento
En la terminal:
```bash
kubectl apply -f k8s/pvc-resultados.yaml
kubectl get pvc
```


## 4. Ejecutar el ETL con Kubernetes
En la terminal:
```bash
kubectl apply -f k8s/job-etl.yaml
kubectl get jobs
kubectl get pods
```

**¿Qué sucede?** Kubernetes crea un Pod para el Job y ejecuta `python etl_paralelo.py` dentro del contenedor. El Job debe terminar con una ejecución completada.

### Revisar el resultado y los registros
En la terminal:
```bash
kubectl logs job/etl-covid-job
```
El Job debe indicar que se completó y los registros deben mostrar la ejecución del ETL. 

**Registro de ejecución:**
- Estado final del Job: Complete
- Filas procesadas: 108733
- Tiempo reportado por el script: 0.4975 segundos

## 5. Recuperar los archivos generados
El Job guarda sus salidas en un PVC. Para recuperarlas usaremos un Pod lector que monte el mismo volumen:
```bash
kubectl apply -f k8s/pod-lector.yaml
kubectl wait --for=condition=Ready pod/etl-lector --timeout=120s
kubectl exec etl-lector -- ls -lh /resultados
mkdir -p resultados_kubernetes
kubectl cp etl-lector:/resultados/COVID19MEXICO_limpio.csv resultados_kubernetes/COVID19MEXICO_limpio.csv
kubectl cp etl-lector:/resultados/auditoria.json resultados_kubernetes/auditoria.json
```

**¿Qué ocurre?** El Pod lector accede al almacenamiento donde el Job guardó los archivos y `kubectl cp` los copia 

## 6. Comprobar la auditoría desde Python

In [3]:
from pathlib import Path
import json

# Ruta del archivo de auditoría que copié desde Kubernetes.
ruta_auditoria = Path("resultados_kubernetes/auditoria.json")

# Solo intento leer el archivo si realmente existe.
if ruta_auditoria.exists():
    # Abro el JSON como texto y lo convierto a un diccionario de Python.
    with ruta_auditoria.open("r", encoding="utf-8") as archivo:
        auditoria = json.load(archivo)
    print(json.dumps(auditoria, indent=4, ensure_ascii=False))
else:
    print("Todavía no se encuentra la auditoría. Primero debo ejecutar el Job y copiar los resultados.")

{
    "filas_entrada": 108733,
    "duplicados_exactos_eliminados": 0,
    "duplicados_id_eliminados": 0,
    "edades_invalidas_convertidas_a_nulo": 0,
    "filas_salida": 108733,
    "procesos_utilizados": 4,
    "tiempo_segundos": 0.49747971200031316
}


## 7. Comprobar el CSV resultante.

In [4]:
from pathlib import Path
import pandas as pd

ruta_csv = Path("resultados_kubernetes/COVID19MEXICO_limpio.csv")

if ruta_csv.exists():
    # Leo el archivo generado por el ETL que se ejecutó en Kubernetes.
    datos_limpios = pd.read_csv(ruta_csv)

    # Muestro filas y columnas para registrar el resultado.
    print("Filas y columnas:", datos_limpios.shape)
    display(datos_limpios.head())
else:
    print("Todavía no se encuentra el CSV. Debo copiarlo desde Kubernetes.")

Filas y columnas: (108733, 42)


,FECHA_ACTUALIZACION,ID_REGISTRO,ORIGEN,SECTOR,ENTIDAD_UM,SEXO,ENTIDAD_NAC,ENTIDAD_RES,MUNICIPIO_RES,TIPO_PACIENTE,...,RESULTADO_PCR,RESULTADO_PCR_COINFECCION,TOMA_MUESTRA_ANTIGENO,RESULTADO_ANTIGENO,CLASIFICACION_FINAL_COVID,CLASIFICACION_FINAL_FLU,MIGRANTE,PAIS_NACIONALIDAD,PAIS_ORIGEN,UCI
0,2026-11-02,g829cab,1,12,11,2,11,11,15,1,...,997,997,2,97,6,6,99,México,97,97
1,2026-11-02,gc633cc,1,8,9,2,7,9,12,1,...,997,997,2,97,6,6,99,México,97,97
2,2026-11-02,gad762a,1,4,30,2,30,30,87,2,...,3,998,2,97,7,3,99,México,97,2
3,2026-11-02,gd88222,1,8,9,2,9,9,10,1,...,997,997,2,97,6,6,99,México,97,97
4,2026-11-02,g8447ee,1,4,9,2,9,9,6,2,...,997,997,2,97,6,6,99,México,97,2
